# (IIP314W) Optimización Aplicada a Negocios
## Ayudantía 8: Problema Dual, Simplex Dual, Holguras Complementarias y Análisis de Sensibilidad

---

**Profesor:** Ing. Rodrigo Trigo Vilches  
**Ayudante:** Lic. Vicente Ramírez Almonacid  
**Fecha:** 29 de Abril, 2026  
**Universidad del Desarrollo**

---

## Sección 0 — Resumen Teórico

Esta ayudantía consolida cuatro conceptos que se desprenden naturalmente del Método Simplex: el **Problema Dual**, el **Algoritmo Simplex Dual**, las **Condiciones de Holgura Complementaria** y el **Análisis de Sensibilidad**. Todos ellos son herramientas que permiten extraer información adicional de la solución óptima sin resolver el problema desde cero.

### Tema 1: El Problema Dual

#### 1.1 Par Primal–Dual en Forma Canónica

A todo problema de programación lineal (**primal**) le corresponde un problema **dual** asociado. En su forma canónica de minimización:

| | Primal | Dual |
|:---:|:---|:---|
| **Objetivo** | $\min \quad c^\top x$ | $\max \quad b^\top y$ |
| **Restricciones** | $Ax \geq b$ | $A^\top y \leq c$ |
| **No negatividad** | $x \geq 0$ | $y \geq 0$ |

#### 1.2 Reglas de Construcción del Dual

Las siguientes reglas permiten construir el dual directamente desde el primal, sin necesidad de memorizar la forma canónica:

| Primal ($\min$) | Dual ($\max$) |
|:---|:---|
| Restricción $i$: $\geq b_i$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\leq b_i$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j \leq 0$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j$ libre | Restricción $j$: $= c_j$ |

> **Nota práctica:** el primal de maximización con restricciones $\leq$ se transforma directamente en el dual de minimización con restricciones $\geq$, y viceversa. Siempre verificar que el número de variables del dual igual al número de restricciones del primal.

#### 1.3 Teorema de Dualidad Débil

Para cualquier $x$ factible en el primal y cualquier $y$ factible en el dual:

$$b^\top y \leq c^\top x$$

Esto implica que el valor objetivo del dual es siempre una **cota inferior** del valor objetivo del primal (en el caso min/max). En particular, si se encuentra un par $(x, y)$ factible tal que $b^\top y = c^\top x$, ambos son óptimos.

#### 1.4 Teorema de Dualidad Fuerte

Si el primal tiene solución óptima $x^*$, entonces el dual también tiene solución óptima $y^*$, y se cumple:

$$c^\top x^* = b^\top y^*$$

Es decir, en el óptimo los valores de ambos objetivos **coinciden exactamente**.

#### 1.5 Interpretación Económica: Precios Sombra

Las variables duales $y_i^*$ se denominan **precios sombra** (o precios duales) de las restricciones. Cada $y_i^*$ representa el **valor marginal** de relajar la restricción $i$ en una unidad:

$$y_i^* = \frac{\partial z^*}{\partial b_i}$$

Si la restricción $i$ está **inactiva** (holgura positiva), su precio sombra es $y_i^* = 0$: relajar esa restricción no mejora el óptimo. Si está **activa** (holgura cero), $y_i^* > 0$: incrementar $b_i$ en una unidad mejora $z^*$ en exactamente $y_i^*$ unidades.

#### Tabla de Relaciones de Dualidad en Programación Lineal

La siguiente tabla resume **todas** las correspondencias primal–dual, cubriendo los dos casos canónicos (primal de maximización y primal de minimización):

| | **Primal MAX** | **Dual MIN** |
|:---:|:---:|:---:|
| **Objetivo** | $\max\ c^\top x$ | $\min\ b^\top y$ |
| Restricción $i$: $\leq b_i$ | $\Leftrightarrow$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\geq b_i$ | $\Leftrightarrow$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | $\Leftrightarrow$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | $\Leftrightarrow$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j \leq 0$ | $\Leftrightarrow$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j$ libre | $\Leftrightarrow$ | Restricción $j$: $= c_j$ |

| | **Primal MIN** | **Dual MAX** |
|:---:|:---:|:---:|
| **Objetivo** | $\min\ c^\top x$ | $\max\ b^\top y$ |
| Restricción $i$: $\geq b_i$ | $\Leftrightarrow$ | Variable $y_i \geq 0$ |
| Restricción $i$: $\leq b_i$ | $\Leftrightarrow$ | Variable $y_i \leq 0$ |
| Restricción $i$: $= b_i$ | $\Leftrightarrow$ | Variable $y_i$ libre |
| Variable $x_j \geq 0$ | $\Leftrightarrow$ | Restricción $j$: $\leq c_j$ |
| Variable $x_j \leq 0$ | $\Leftrightarrow$ | Restricción $j$: $\geq c_j$ |
| Variable $x_j$ libre | $\Leftrightarrow$ | Restricción $j$: $= c_j$ |

> **Regla nemotécnica:** al pasar de MAX a MIN (o viceversa), el sentido de las desigualdades de restricciones y el signo de las variables se **invierten**. La relación es simétrica: el dual del dual es el primal.

### Tema 2: Algoritmo Simplex Dual

#### 2.1 Diferencia Clave Respecto al Simplex Primal

El **Simplex Primal** parte de una solución *factible* (RHS $\geq 0$) y busca la *optimalidad* (costos reducidos $\geq 0$).  
El **Simplex Dual** parte de una solución *dual-factible* (costos reducidos $\geq 0$ ya satisfechos) pero *primal-infactible* (algún RHS $< 0$), y trabaja para recuperar la factibilidad.

Esto ocurre naturalmente cuando un problema de minimización con restricciones $\geq$ se plantea directamente en tableau (las holguras son negativas).

#### 2.2 Reglas de Pivoteo Dual

| Paso | Criterio |
|:---:|:---|
| **Variable que sale** | La variable básica con el RHS más negativo (fila más infactible). |
| **Variable que entra** | La variable no básica que minimiza el cociente $\left|\dfrac{\bar{c}_j}{a_{rj}}\right|$ para las columnas con $a_{rj} < 0$ en la fila pivote $r$. |
| **Optimalidad** | Se alcanza cuando todos los RHS son $\geq 0$ (la solución se vuelve factible). |
| **Infactibilidad** | Si en alguna iteración una fila pivote tiene todos sus coeficientes $\geq 0$ pero RHS $< 0$, el problema primal es **infactible**. |

#### 2.3 Tableau del Simplex Dual

El tableau es el mismo que el del Simplex Primal. La diferencia está en **qué elemento se elige como pivote**:

$$\text{Entra: } \arg\min_{j:\, a_{rj}<0} \left|\frac{\bar{c}_j}{a_{rj}}\right| \qquad \text{Sale: } r = \arg\min_i \{\bar{b}_i \mid \bar{b}_i < 0\}$$

> La condición de entrada busca preservar la **dual-factibilidad**: que los costos reducidos permanezcan $\geq 0$ tras el pivoteo.

### Tema 3: Condiciones de Holgura Complementaria

#### 3.1 Enunciado

Para cualquier par de soluciones óptimas $x^*$ (primal) e $y^*$ (dual), se cumplen las siguientes condiciones para **todo** $i$ y $j$:

$$y_i^* \cdot \underbrace{\left(a_i^\top x^* - b_i\right)}_{\text{holgura primal}} = 0 \qquad \forall i$$

$$x_j^* \cdot \underbrace{\left(c_j - a_j^\top y^*\right)}_{\text{holgura dual}} = 0 \qquad \forall j$$

En palabras:
- Si la restricción primal $i$ **no está activa** ($a_i^\top x^* > b_i$), entonces $y_i^* = 0$.
- Si $y_i^* > 0$, entonces la restricción primal $i$ **debe estar activa** ($a_i^\top x^* = b_i$).
- Si $x_j^* > 0$, entonces la restricción dual $j$ **debe estar activa** ($a_j^\top y^* = c_j$).
- Si la restricción dual $j$ **no está activa** ($a_j^\top y^* < c_j$), entonces $x_j^* = 0$.

#### 3.2 Uso Práctico: Recuperar la Solución Primal desde el Dual

Si se conoce la solución óptima del dual $y^*$:

1. Identificar qué $y_i^* > 0$: las restricciones primales correspondientes son activas ($=b_i$).
2. Identificar qué $y_i^* = 0$: las restricciones correspondientes pueden tener holgura.
3. Usar las restricciones activas como sistema de ecuaciones para encontrar $x^*$.
4. Verificar que $x^* \geq 0$ y que las restricciones inactivas no sean violadas.

### Tema 4: Análisis de Sensibilidad

El análisis de sensibilidad responde a la pregunta: **¿en qué rango puede variar un parámetro del problema sin que la base óptima actual deje de ser óptima?**

Sea $B$ la base óptima con $B^{-1}$ conocida.

#### 4.1 Sensibilidad sobre $b$ (lado derecho / RHS)

Si se perturba el RHS como $b \to b + \Delta e_i$ (modificar $b_i$ en $\Delta$), la solución básica actualizada es:

$$x_B = B^{-1}(b + \Delta e_i) = B^{-1}b + \Delta B^{-1}e_i = \bar{b} + \Delta d_i$$

donde $d_i$ es la $i$-ésima columna de $B^{-1}$. La base sigue siendo factible mientras:

$$\bar{b} + \Delta d_i \geq 0 \qquad \Rightarrow \qquad \Delta \geq -\frac{\bar{b}_k}{(d_i)_k} \text{ para } (d_i)_k > 0 \quad \text{ y } \quad \Delta \leq -\frac{\bar{b}_k}{(d_i)_k} \text{ para } (d_i)_k < 0$$

El **precio sombra** de la restricción $i$ es $y_i^* = c_B^\top B^{-1} e_i$, y representa la tasa de cambio de $z^*$ con respecto a $b_i$.

#### 4.2 Sensibilidad sobre $c$ (coeficientes del objetivo)

Si se perturba el coeficiente objetivo de la variable **no básica** $x_j$ como $c_j \to c_j + \Delta$, el costo reducido de $x_j$ cambia:

$$\bar{c}_j + \Delta \geq 0 \qquad \Rightarrow \qquad \Delta \geq -\bar{c}_j$$

Si se perturba el coeficiente de la variable **básica** $x_k$ (que está en la posición $p$ de la base), el vector $c_B$ cambia y todos los costos reducidos de las no básicas se ven afectados:

$$\bar{c}_j^{\text{nuevo}} = \bar{c}_j - \Delta (B^{-1}A_{NB})_{p,j} \geq 0 \qquad \forall j \notin B$$

Esto define un sistema de inecuaciones en $\Delta$ cuya intersección es el rango de optimalidad.

#### 4.3 Interpretación del Precio Sombra

El precio sombra $y_i^*$ de la restricción $i$ tiene unidades de **[unidades de objetivo] / [unidades de $b_i$]**. Por ejemplo, si $b_i$ es horas disponibles y $z$ es beneficio en pesos, entonces $y_i^*$ es el beneficio adicional por hora extra. Este valor es válido **solo dentro del rango de sensibilidad** de $b_i$.

---

### Ejemplo Ilustrativo: Construcción del Par Primal–Dual

Consideremos el siguiente problema sencillo para ilustrar cómo se construye el dual paso a paso.

**Problema Primal:**

$$\max \quad z = 2x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases} x_1 + x_2 \leq 4 \\ x_1 + 2x_2 \leq 6 \\ x_1,\, x_2 \geq 0 \end{cases}$$

**Paso 1 — Identificar el tipo de problema y restricciones:**

- Objetivo: maximización → el dual será de **minimización**
- 2 restricciones $\leq$ → 2 variables duales $y_1, y_2 \geq 0$
- 2 variables $x_j \geq 0$ → 2 restricciones duales de tipo $\geq$

**Paso 2 — Construir la función objetivo dual** (coeficientes del RHS del primal):

$$\min \quad w = 4y_1 + 6y_2$$

**Paso 3 — Construir las restricciones duales** (una por cada variable primal; los coeficientes se leen por **columnas** de la matriz $A$):

| Variable primal | Columna en $A$ | Restricción dual | RHS |
|:---:|:---:|:---:|:---:|
| $x_1$ | $(1,\; 1)$ | $y_1 + y_2 \geq$ | $2$ |
| $x_2$ | $(1,\; 2)$ | $y_1 + 2y_2 \geq$ | $3$ |

**Problema Dual resultante:**

$$\min \quad w = 4y_1 + 6y_2$$

$$\text{s.a.} \quad \begin{cases} y_1 + y_2 \geq 2 \\ y_1 + 2y_2 \geq 3 \\ y_1,\, y_2 \geq 0 \end{cases}$$

**Paso 4 — Verificar la dualidad fuerte** (resolución manual):

*Primal:* en el óptimo, ambas restricciones están activas (vértice del poliedro):
$$x_1 + x_2 = 4 \quad \text{y} \quad x_1 + 2x_2 = 6 \quad \Rightarrow \quad x_1^* = 2,\; x_2^* = 2, \quad z^* = 2(2)+3(2) = \mathbf{10}$$

*Dual:* por holguras complementarias, como $x_1^*, x_2^* > 0$, ambas restricciones duales son activas:
$$y_1 + y_2 = 2 \quad \text{y} \quad y_1 + 2y_2 = 3 \quad \Rightarrow \quad y_1^* = 1,\; y_2^* = 1, \quad w^* = 4(1)+6(1) = \mathbf{10}$$

$$\boxed{z^* = w^* = 10 \quad \checkmark \text{ Dualidad Fuerte}}$$

> **Lectura económica:** $y_1^* = 1$ indica que una unidad extra de la primera restricción (capacidad 4) aumenta el valor óptimo en $1$. Análogamente para $y_2^* = 1$ con la segunda restricción (capacidad 6).

---

---

## Ejercicio 1 — Planteamiento del Dual y Resolución

### Contexto de Negocio

**TechPrint S.A.** es una empresa de impresión digital que produce dos tipos de materiales publicitarios: **Afiches** ($x_1$, en cientos de unidades) y **Catálogos** ($x_2$, en cientos de unidades). La empresa opera con tres recursos limitados semanalmente:

| Recurso | Afiches ($x_1$) | Catálogos ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Tinta especial (litros) | $2$ | $3$ | $\leq 18$ L |
| Tiempo de prensa (horas) | $4$ | $2$ | $\leq 20$ hrs |
| Papel recubierto (resmas) | $1$ | $3$ | $\leq 15$ resmas |
| **Margen neto (\$/cien unid.)** | **\$5** | **\$4** | — |

El gerente de producción desea maximizar el margen neto semanal.

### Formulación Primal

$$\max \quad z = 5x_1 + 4x_2$$

$$\text{s.a.} \quad \begin{cases}
2x_1 + 3x_2 \leq 18 & \text{(tinta)} \\
4x_1 + 2x_2 \leq 20 & \text{(prensa)} \\
x_1 + 3x_2 \leq 15 & \text{(papel)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

### Parte (a): Plantee el Problema Dual

Usando la tabla de reglas de dualidad, construya el problema dual del primal anterior.

- ¿Cuántas variables tendrá el dual? ¿De qué signo serán?
- ¿Cuántas restricciones tendrá el dual? ¿De qué tipo serán?
<!-- - ¿El dual es de maximización o minimización? -->

### Parte (b): Resuelva el Dual por Simplex Dual (Tableau)

El dual planteado es de minimización con restricciones $\geq$. Convierta las restricciones a la forma estándar del Simplex Dual (multiplicando por $-1$) y resuelva iterativamente.

**Forma estándar del Dual** (con excedentes $s_1, s_2$ y filas multiplicadas por $-1$):

### Parte (c): Recupere la Solución Primal usando Holguras Complementarias

Dado que conoce $y_1^*$, $y_2^*$ e $y_3^*$, utilice las condiciones de holgura complementaria para determinar $x_1^*$ y $x_2^*$ sin resolver el primal.

### Parte (d): Verifique la Dualidad Fuerte

Calcule $z^*$ y $w^*$ y verifique que coinciden.

---

## Ejercicio 2 — Análisis de Sensibilidad

### Contexto de Negocio

**AgroNorte Ltda.** es una empresa agroindustrial que procesa y comercializa dos tipos de conservas: **Conserva de Tomate** ($x_1$, en toneladas) y **Conserva de Pimiento** ($x_2$, en toneladas). El proceso productivo está limitado por la capacidad de dos líneas de procesamiento:

| Recurso | Tomate ($x_1$) | Pimiento ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Línea de esterilización (hrs/ton) | $3$ | $2$ | $\leq 12$ horas |
| Línea de envasado (hrs/ton) | $1$ | $2$ | $\leq 8$ horas |
| **Margen neto (M\$/ton)** | **\$6** | **\$5** | — |

La empresa ha resuelto el siguiente modelo primal:

$$\max \quad z = 6x_1 + 5x_2$$

$$\text{s.a.} \quad \begin{cases}
3x_1 + 2x_2 \leq 12 & \text{(esterilización)} \\
x_1 + 2x_2 \leq 8 & \text{(envasado)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

Resolviendo por el Método Simplex, la solución óptima se obtiene con la base $B = \{x_1, x_2\}$. El **tableau óptimo** ya resuelto es:

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $0$ | $0$ | $7/4$ | $5/4$ | $\mathbf{26}$ |
| $x_1$ | $1$ | $0$ | $1/2$ | $-1/2$ | $2$ |
| $x_2$ | $0$ | $1$ | $-3/8$ | $3/4$ | $3$ |

La solución óptima es $x_1^* = 2$ ton Tomate, $x_2^* = 3$ ton Pimiento, $z^* = \$26$ M.

**Datos para el análisis:**

$$B = \begin{bmatrix} 3 & 2 \\ 1 & 2 \end{bmatrix}, \quad B^{-1} = \begin{bmatrix} 1/2 & -1/2 \\ -1/4 & 3/4 \end{bmatrix}, \quad c_B = [6,\; 5], \quad b = \begin{bmatrix} 12 \\ 8 \end{bmatrix}$$

> Recuerde: los precios sombra aparecen en la fila CR del tableau óptimo en las columnas de las holguras: $y_1^* = 7/4$ y $y_2^* = 5/4$.

### Parte (a): Análisis de Sensibilidad sobre $b_1$ (horas de esterilización)

¿Cuánto puede variar $b_1 = 12$ horas sin que cambie la base óptima $\{x_1, x_2\}$?

Sea $b_1 \to 12 + \Delta$. La solución básica actualizada es:

$$x_B = B^{-1}b + \Delta B^{-1}e_1 = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \cdot \underbrace{\begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix}}_{\text{1a col. de }B^{-1}}$$

**Condición de factibilidad** ($x_B \geq 0$):

$$x_1 = 2 + \underline{\quad} \cdot \Delta \geq 0 \quad \Rightarrow \quad \Delta \geq \underline{\quad}$$

$$x_2 = 3 + \underline{\quad} \cdot \Delta \geq 0 \quad \Rightarrow \quad \Delta \leq \underline{\quad}$$

**Rango de optimalidad para $b_1$:**

$$\underline{\quad} \leq \Delta \leq \underline{\quad} \quad \Rightarrow \quad \boxed{\underline{\quad} \leq b_1 \leq \underline{\quad}}$$

**Interpretación en el contexto de AgroNorte:**

> 📝 Respuesta:

### Parte (b): Análisis de Sensibilidad sobre $b_2$ (horas de envasado)

¿Cuánto puede variar $b_2 = 8$ horas sin que cambie la base óptima?

Sea $b_2 \to 8 + \Delta$. La solución básica actualizada es:

$$x_B = B^{-1}b + \Delta B^{-1}e_2 = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \cdot \underbrace{\begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix}}_{\text{2a col. de }B^{-1}}$$

**Condición de factibilidad** ($x_B \geq 0$):

$$x_1 = 2 + \underline{\quad} \cdot \Delta \geq 0 \quad \Rightarrow \quad \Delta \underline{\quad}$$

$$x_2 = 3 + \underline{\quad} \cdot \Delta \geq 0 \quad \Rightarrow \quad \Delta \underline{\quad}$$

**Rango de optimalidad para $b_2$:**

$$\boxed{\underline{\quad} \leq b_2 \leq \underline{\quad}}$$

**Interpretación en el contexto de AgroNorte:**

> 📝 Respuesta:

### Parte (c): Análisis de Sensibilidad sobre $c_1$ (margen de Conserva de Tomate)

¿En qué rango puede variar el margen neto de la Conserva de Tomate ($c_1 = 6$) sin que cambie la base óptima $\{x_1, x_2\}$?

Como $x_1$ es **variable básica**, cambiar $c_1$ afecta a $c_B = [6+\Delta, 5]$ y modifica los costos reducidos de las no básicas ($s_1$ y $s_2$).

**Costo reducido de $s_1$:**

$$\bar{c}_{s_1} = [6+\Delta,\; 5] \cdot \begin{bmatrix}1/2\\-1/4\end{bmatrix} - 0 = \underline{\hspace{5cm}}$$

**Costo reducido de $s_2$:**

$$\bar{c}_{s_2} = [6+\Delta,\; 5] \cdot \begin{bmatrix}-1/2\\3/4\end{bmatrix} - 0 = \underline{\hspace{5cm}}$$

**Condición de optimalidad** ($\bar{c}_{s_1} \geq 0$ y $\bar{c}_{s_2} \geq 0$):

$$\underline{\hspace{4cm}} \geq 0 \quad \Rightarrow \quad \Delta \geq \underline{\quad}$$

$$\underline{\hspace{4cm}} \geq 0 \quad \Rightarrow \quad \Delta \leq \underline{\quad}$$

**Rango de optimalidad para $c_1$:**

$$\boxed{\underline{\quad} \leq c_1 \leq \underline{\quad}}$$

**Interpretación en el contexto de AgroNorte:**

> 📝 Respuesta:

### Parte (d): Interpretación de los Precios Sombra en el Contexto de AgroNorte

Los precios sombra del tableau óptimo son $y_1^* = 7/4 = 1{,}75$ M\$/hora y $y_2^* = 5/4 = 1{,}25$ M\$/hora.

Responda las siguientes preguntas en términos del negocio:

1. ¿Qué significa concretamente que $y_1^* = 1{,}75$ M\$/hora para la gerencia de AgroNorte?

> 📝 Respuesta:

2. ¿Cuál de las dos líneas de procesamiento es el cuello de botella más valioso y por qué?

> 📝 Respuesta:

3. Si AgroNorte puede arrendar capacidad adicional de esterilización a un costo de \$1,50 M/hora, ¿le conviene hacerlo? ¿Cuántas horas adicionales puede arrendar manteniendo válido el análisis?

> 📝 Respuesta:

4. ¿En qué rango de $b_1$ y $b_2$ son válidos los precios sombra calculados?

> 📝 Respuesta:

---

## Ejercicio 3 — Dual Completo y Sensibilidad Combinada

### Contexto de Negocio

**LogiCarga S.A.** es una empresa de logística urbana que reparte pedidos usando dos tipos de vehículos: **Furgones** ($x_1$, en decenas de viajes diarios) y **Motos** ($x_2$, en decenas de viajes diarios). La empresa quiere **minimizar el costo operacional diario** sujeto a compromisos mínimos de servicio:

| Condición | Furgones ($x_1$) | Motos ($x_2$) | Requerimiento |
|:---|:---:|:---:|:---:|
| Pedidos grandes cubiertos (uds/decena viajes) | $3$ | $1$ | $\geq 9$ unidades |
| Pedidos urgentes cubiertos (uds/decena viajes) | $1$ | $2$ | $\geq 8$ unidades |
| **Costo (M\$/decena de viajes)** | **\$4** | **\$3** | — |

### Formulación Primal

$$\min \quad z = 4x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases}
3x_1 + x_2 \geq 9 & \text{(pedidos grandes)} \\
x_1 + 2x_2 \geq 8 & \text{(pedidos urgentes)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

### Parte (a): Plantee el Dual

Construya el dual de este problema de minimización con restricciones $\geq$.

$$\text{(tipo de objetivo)} \quad w = \underline{\hspace{5cm}}$$

$$\text{s.a.} \quad \begin{cases}
\underline{\hspace{6cm}} & \text{(restricción para } x_1\text{)} \\
\underline{\hspace{6cm}} & \text{(restricción para } x_2\text{)} \\
y_1,\, y_2 \geq 0
\end{cases}$$

> 📝 Respuesta:

### Parte (b): Resuelva el Primal por Simplex Dual (Tableau)

**Forme el tableau inicial** (con filas multiplicadas por $-1$ para las restricciones):

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | | | | | |
| $s_1$ | | | | | |
| $s_2$ | | | | | |

**Iteración 0 → Iteración 1:**

Sale: __________ (RHS más negativo) | Entra: __________ (cociente $|z_j/a_{rj}|$ mínimo)

Pivote: __________ | Operaciones: _(escribe aquí las operaciones de fila)_

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | | | | | |
| | | | | | |
| | | | | | |

**Iteración 1 → Iteración 2:**

Sale: __________ | Entra: __________ | Pivote: __________

_(escribe aquí las operaciones de fila)_

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **$z$** | | | | | |
| | | | | | |
| | | | | | |

¿Se cumple la condición de optimalidad? _____ ¿Por qué? _______________________

$$\boxed{x_1^* = \underline{\quad}, \quad x_2^* = \underline{\quad}, \quad z^* = \underline{\quad} \text{ M\$}}$$

> 📝 Respuesta:

### Parte (c): Análisis de Sensibilidad sobre $b_1$ (pedidos grandes)

Del tableau óptimo, la base óptima es $B = \{x_1, x_2\}$ con:

$$B = \begin{bmatrix} 3 & 1 \\ 1 & 2 \end{bmatrix}, \qquad B^{-1} = \begin{bmatrix} 2/5 & -1/5 \\ -1/5 & 3/5 \end{bmatrix}$$

Encuentre el rango de $b_1 = 9$ pedidos grandes tal que la base actual siga siendo óptima.

Sea $b_1 \to 9 + \Delta$:

$$x_B = \begin{bmatrix}2\\3\end{bmatrix} + \Delta \cdot \begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix}$$

**Condición de factibilidad:**

$$x_1 = \underline{\hspace{4cm}} \geq 0 \quad \Rightarrow \quad \Delta \geq \underline{\quad}$$

$$x_2 = \underline{\hspace{4cm}} \geq 0 \quad \Rightarrow \quad \Delta \leq \underline{\quad}$$

$$\boxed{\underline{\quad} \leq b_1 \leq \underline{\quad}}$$

**Precio sombra $y_1^*$** (calcule $c_B^\top (B^{-1})_{\text{col }1}$):

$$y_1^* = [4,\; 3] \cdot \begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix} = \underline{\quad} \text{ M\$/unidad de pedido grande}$$

**Interpretación:**

> 📝 Respuesta:

### Parte (d): Verificación de Dualidad Fuerte y Holguras Complementarias

Los precios sombra del primal corresponden a los valores duales óptimos. Del tableau óptimo, identifique $y_1^*$ e $y_2^*$ (aparecen en la fila $z$ en las columnas de $s_1$ y $s_2$).

**Solución dual:**

$$y_1^* = \underline{\quad}, \qquad y_2^* = \underline{\quad}$$

**Verificación de factibilidad dual** ($3y_1 + y_2 \leq 4$ y $y_1 + 2y_2 \leq 3$):

$$3(\underline{\quad}) + (\underline{\quad}) = \underline{\quad} \leq 4 \; \text{✓/✗}$$
$$\underline{\quad} + 2(\underline{\quad}) = \underline{\quad} \leq 3 \; \text{✓/✗}$$

**Dualidad fuerte:**

$$z^* = 4(\underline{\quad}) + 3(\underline{\quad}) = \underline{\quad}$$
$$w^* = 9(\underline{\quad}) + 8(\underline{\quad}) = \underline{\quad}$$

$$z^* \;\underline{\;=\;/\;\neq\;}\; w^* \quad \text{(✓/✗ Dualidad Fuerte)}$$

**Holguras complementarias** — complete la tabla:

| Restricción | ¿Activa? | $y_i^*$ | ¿Consistente con HC? |
|:---|:---:|:---:|:---:|
| Pedidos grandes: $3x_1^* + x_2^* = $ _____ | ✓/✗ | $y_1^* = $ _____ | ✓/✗ |
| Pedidos urgentes: $x_1^* + 2x_2^* = $ _____ | ✓/✗ | $y_2^* = $ _____ | ✓/✗ |

**Conclusión operacional de LogiCarga:**

> 📝 Respuesta:

---

## Resumen de la Ayudantía

| Concepto | Idea Clave | Herramienta |
|:---|:---|:---|
| **Problema Dual** | A todo primal le corresponde un dual; variables duales son precios sombra | Tabla de dualidad |
| **Dualidad Débil** | $b^\top y \leq c^\top x$ para cualquier par factible | Cota |
| **Dualidad Fuerte** | $z^* = w^*$ en el óptimo | Verificación |
| **Simplex Dual** | Sale el RHS más negativo; entra el cociente $|w_j / a_{rj}|$ mínimo | Tableau |
| **Holguras Complementarias** | $y_i^* (a_i^\top x^* - b_i) = 0$ y $x_j^* (c_j - a_j^\top y^*) = 0$ | Recuperar primal desde dual |
| **Sensibilidad en $b$** | Rango de $\Delta$ tal que $B^{-1}(b + \Delta e_i) \geq 0$ | Análisis post-óptimo |
| **Sensibilidad en $c$** | Rango de $\Delta$ tal que todos $\bar{c}_j \geq 0$ con nuevo $c_k$ | Análisis post-óptimo |
| **Precio Sombra** | $y_i^* = \partial z^* / \partial b_i$ — válido solo en el rango de sensibilidad | Decisiones de inversión |

---

*Ayudantía 8 — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T1*